# aiayn: Kaggle validation

Validates a Kaggle session before any training (docs/KAGGLE.md): environment, dependencies, CUDA,
dataset access, tokenizer, model construction, forward/backward, **VRAM** and **throughput**.
Target runtime: under 10 minutes.

**Session options:** Accelerator **GPU T4**. Two ways to get the code:

* `CODE_SOURCE = "github"` (default): clones the GitHub repository. Needs **Internet ON**.
  Multi30k is downloaded (checksum-verified) and the tokenizer is trained here, then compared with
  the fingerprint of the tokenizer used locally.
* `CODE_SOURCE = "dataset"`: offline; attach the `aiayn-code` and `aiayn-multi30k` datasets
  built by `scripts/package_for_kaggle.py`.

All logic lives in the repository (`scripts/kaggle_validate.py`, `aiayn.diagnostics`).

In [ ]:
# 0. Parameters
CODE_SOURCE = "github"  # "github" or "dataset"
GITHUB_URL = (
    "https://github.com/mohamed242006/Transformer-from-scratch-with-a-paper-reproduction.git"
)
GITHUB_BRANCH = "master"
CODE_DATASET = "/kaggle/input/aiayn-code"  # used when CODE_SOURCE == "dataset"
DATA_DATASET = "/kaggle/input/aiayn-multi30k"  # optional offline data + tokenizer bundle
# Tokenizer fingerprint recorded on the development machine (Week 2, Multi30k BPE 8k).
EXPECTED_TOKENIZER_FINGERPRINT = "f04bec5925700e0aedbfd3e700e67823b3a59f0bdd17d5e01ab7a9f9f07c930a"
WORK = "/kaggle/working"
MODEL_CONFIGS = [
    "configs/model/transformer_tiny.yaml",
    "configs/model/transformer_iwslt.yaml",
]
THROUGHPUT_STEPS = 30
RUN_PREFLIGHT_TESTS = True  # policy + parity + numerical suites (about 1 min)

In [ ]:
# 1. Get the code into a writable directory
import glob
import importlib.util
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile

import torch

T0 = time.time()
TORCH_BEFORE = torch.__version__  # must be unchanged after installation
REPO = f"{WORK}/aiayn"
shutil.rmtree(REPO, ignore_errors=True)
if CODE_SOURCE == "github":
    clone = ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_URL, REPO]
    subprocess.run(clone, check=True)
    head = subprocess.run(
        ["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True, check=True
    )
    CODE_VERSION = "git " + head.stdout.strip()
else:
    found = glob.glob(f"{CODE_DATASET}/**/pyproject.toml", recursive=True)
    if not found:  # the upload may still be a zip archive
        archives = glob.glob(f"{CODE_DATASET}/**/*.zip", recursive=True)
        assert archives, f"no pyproject.toml or .zip under {CODE_DATASET}"
        with zipfile.ZipFile(archives[0]) as zf:
            zf.extractall(f"{WORK}/_code_unzipped")
        found = glob.glob(f"{WORK}/_code_unzipped/**/pyproject.toml", recursive=True)
    shutil.copytree(os.path.dirname(min(found, key=len)), REPO)
    with open(f"{REPO}/AIAYN_MANIFEST.json") as fh:
        CODE_VERSION = "tree " + json.load(fh)["tree_sha256"]
os.chdir(REPO)
assert os.path.exists("scripts/kaggle_validate.py"), (
    "validator missing: wrong branch or stale code?"
)
print("code:", REPO, "|", CODE_VERSION, "| torch:", TORCH_BEFORE)

In [ ]:
# 2. Dependencies: install the package without touching Kaggle's torch
def run(cmd, **kw):
    print("$", " ".join(cmd))
    return subprocess.run(cmd, **kw)


ENV = dict(os.environ, PYTHONPATH=f"{REPO}/src")  # fallback if the editable install fails
r = run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])
print("editable install:", "ok" if r.returncode == 0 else "FAILED -> using PYTHONPATH fallback")
needed = {
    "sentencepiece": "sentencepiece>=0.2.0",
    "sacrebleu": "sacrebleu>=2.4",
    "yaml": "pyyaml>=6.0",
    "pytest": "pytest>=8.0",
}
missing = [spec for module, spec in needed.items() if importlib.util.find_spec(module) is None]
if missing:  # none of these depend on torch, so torch cannot be replaced
    run([sys.executable, "-m", "pip", "install", "-q", *missing])
print("missing before install:", missing or "none")

In [ ]:
# 3. Environment snapshot (the validator repeats these checks with pass/fail criteria)
run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"])
print(
    "python",
    sys.version.split()[0],
    "| torch",
    torch.__version__,
    "| cuda",
    torch.version.cuda,
    "| available",
    torch.cuda.is_available(),
)

In [ ]:
# 4. Preflight tests: catch PyTorch numerical drift before spending GPU time (ADR-004, ADR-013)
if RUN_PREFLIGHT_TESTS:
    suites = ["tests/policy", "tests/parity", "tests/numerical"]
    r = run([sys.executable, "-m", "pytest", *suites, "-q", "-p", "no:cacheprovider"], env=ENV)
    assert r.returncode == 0, "preflight tests failed: do not train in this environment"

In [ ]:
# 5. Validation: data, tokenizer, models, forward/backward, VRAM, throughput
REPORT = f"{WORK}/validation/report.json"
cmd = [
    sys.executable,
    "scripts/kaggle_validate.py",
    "--data-config",
    "configs/data/multi30k.yaml",
    "--data-override",
    f"translation.root={WORK}/data",
    "--retrain-tokenizer-check",
    "--require-cuda",
    "--expected-torch-version",
    TORCH_BEFORE,
    "--steps",
    str(THROUGHPUT_STEPS),
    "--output",
    REPORT,
]
bundle = glob.glob(f"{DATA_DATASET}/**/tokenizer_fingerprint.txt", recursive=True)
if bundle:  # offline data bundle: archives as mirror + the packaged tokenizer
    folder = os.path.dirname(bundle[0])
    cmd += ["--data-mirror", folder, "--tokenizer", glob.glob(f"{folder}/*.model")[0]]
    cmd += ["--expected-fingerprint-file", bundle[0]]
else:  # online: download Multi30k (checksummed) and train the tokenizer here
    cmd += [
        "--train-tokenizer-if-missing",
        "--expected-fingerprint",
        EXPECTED_TOKENIZER_FINGERPRINT,
    ]
for config in MODEL_CONFIGS:
    cmd += ["--model-config", config]
VALIDATION_EXIT = run(cmd, env=ENV).returncode

In [ ]:
# 6. Summary (the report JSON is saved with the notebook output)
import pandas as pd

with open(REPORT) as fh:
    report = json.load(fh)
columns = ("name", "status", "critical", "seconds", "message")
table = pd.DataFrame([{k: c[k] for k in columns} for c in report["checks"]])
pd.set_option("display.max_colwidth", 140)
display(table)
for c in report["checks"]:
    if c["name"].endswith((".memory", ".throughput")):
        print(c["name"], json.dumps(c["details"], indent=1))
print(f"code: {CODE_VERSION}")
print(f"notebook wall time: {time.time() - T0:.0f} s")
assert report["ok"] and VALIDATION_EXIT == 0, "VALIDATION FAILED: see the table above"
print("VALIDATION PASSED")